In [19]:
import math
import random
import numpy as np
from collections import Counter

Чтение и приведение текста к нижнему регистру

In [24]:
with open("Достоевский Федор. ИДИОТ.txt", "r", encoding="CP1251") as file:
    text_lower = file.read().lower()


Разбиение на слова регулярным выражением

In [6]:
words = re.findall(r"[a-zа-яё0-9]+(?:-[a-zа-яё0-9]+)*", text_lower)

Статистика

In [10]:
total_words = len(words)
vocab = set(words)
vocab_size = len(vocab)

print(total_words)
print(vocab_size)

word_counts = collections.Counter(words)
top_20_words = word_counts.most_common(20)

print(top_20_words)

206572
27095
[('и', 9831), ('не', 5029), ('в', 4795), ('что', 4371), ('я', 3399), ('с', 2568), ('он', 2522), ('на', 2451), ('как', 1817), ('но', 1765), ('это', 1680), ('а', 1581), ('вы', 1533), ('князь', 1513), ('все', 1506), ('же', 1396), ('так', 1328), ('его', 1319), ('к', 1154), ('бы', 1083)]


In [22]:
split_index = int(len(words) * 0.9)
train_words = words[:split_index]
test_words = words[split_index:]

print(len(train_words))
print(len(test_words))

185914
20658


In [32]:
class BigramModel:
    def __init__(self):
        self.words1 = collections.Counter()
        self.words2  = collections.Counter()
        self.vocab = set()

    def fit(self, tokens):
        self.vocab = set(tokens)
        self.words1 = Counter(tokens)
        self.words2  = Counter(zip(tokens[:-1], tokens[1:]))

    def prob(self, v, w):
        c_v = self.words1.get(v, 0)
        if c_v == 0:
            return 0.0
        return self.words2.get((v, w), 0) / c_v

    def logprob(self, text_tokens):
        total = 0.0
        for v, w in zip(text_tokens[:-1], text_tokens[1:]):
            p = self.prob(v, w)
            if p == 0.0:
                return float("-inf")
            total += math.log(p)
        return total

    def generate(self, start, n, T):
        word = start
        out = [word]
        for i in range(n - 1):
            candidates = [w for (v, w) in self.words2 if v == word]
            if not candidates:
                break
            probs = np.array([self.words2[(word, w)] for w in candidates], dtype=float)
            probs = probs / probs.sum()
            probs = probs ** (1.0 / T)
            probs = probs / probs.sum()
            word = np.random.choice(candidates, p=probs)
            out.append(word)
        return " ".join(out)

In [28]:
model = BigramModel()
model.fit(train_words)

In [36]:
starts = ["князь", "он", "я", "она", "это"]
for s in starts:
    if s in model.words1:
        print(model.generate(s, n=30, T=1.0), '\n')

князь если б князь князь тоже облокотился на лице умершего в тюрьме а какая скверная комната была почти в явный спор даже трех шагах предо мною факта вполне просто и 

он сюда в нем дрожь его ощущение которое признали на крючок иезуитам а он поворотил корабля в том же всю ночь сегодня заключительный отказ на тротуаре скрестив руки чтоб эту 

я не захочет видеть почему-то зачем-то человек отвечай хитрец да ведь вижу что в оправдание на длинных ресницах встань за него и во всяком случае заговорил не штопор ли того 

она много виноват и сказал князь мышкин мы все-таки это вы тоже только я теперь почти всегда мог зная и казалось он чуть в беспокойстве когда он прожил в жизни 

это знаете что надежды полное сознание довольно приличном и до хлеба нет то вы точно князь несмотря на счет лизавета прокофьевна решила про тебя чем сравнительно с претензиями на избранной 



In [49]:
for T in (0.5, 1.0, 2.0):
    for s in starts:
        if s in model.words1:
            print(model.generate(s, n=30, T=T))
    print()

князь лев николаич медленно и как бы в этом роде психологический случай хотя и не знают что его и ужасно рад я вам в этом но я вас князь после
он ужасно поражен до того что предо мной в этом я не знаю что не было бы вам я сам не только что вы на меня так и даже и
я не совсем не было теперь я не знал что она так как бы вы не так и если бы вы думаете что я и не мог бы не было
она теперь не стать я в своем относились об этом роде но он и не покажете этой подушке и по крайней мере вошли в самом деле им нарочно не было
это было довольно я ведь это было что я не в высшей степени буржуазному но когда можно сказать то что я и с ними и с ней меня я бы

князь что я думал она приобрести такие наклонности довольно почти успокоился за окно вагон продолжает рапортовать что и будущностью все это поразительное для передачи-с от иезуита вставила генеральша вышла особенная
он не только думаю усталости авторитетно промямлил князь щ и нормальную дорогу смягчить и не сказал пояснил докторенко в угол князь 

In [51]:
# Связный фрагмент из теста + те же слова в случайном порядке
sentence = test_words[:20]
shuffled = sentence.copy()
random.shuffle(shuffled)

print("Связный:")
print(" ".join(sentence))
print("Перемешанный:")
print(" ".join(shuffled))

lp_conn = model.logprob(sentence)
lp_shuf = model.logprob(shuffled)
print("log P без сглаживания, связный:", lp_conn)
print("log P без сглаживания, перемешанный:", lp_shuf)

Связный:
она и обеспечена у вари но это все не то он вскочил вспомнив что его ждут наскоро спросил о состоянии
Перемешанный:
наскоро все не вари вскочил вспомнив и у его она что спросил о он обеспечена это то но состоянии ждут
log P без сглаживания, связный: -inf
log P без сглаживания, перемешанный: -inf


In [53]:
def logprob_smoothed(model, text_tokens, alpha):
    V = len(model.vocab)
    total = 0.0
    for v, w in zip(text_tokens[:-1], text_tokens[1:]):
        c_v  = model.words1.get(v, 0)
        c_vw = model.words2.get((v, w), 0)
        p = (c_vw + alpha) / (c_v + alpha * V)
        total += math.log(p)
    return total

alpha_demo = 0.1
print("Со сглаживанием связный:",
      logprob_smoothed(model, sentence, alpha_demo))
print("Со сглаживанием перемешанный:",
      logprob_smoothed(model, shuffled, alpha_demo))

Со сглаживанием связный: -153.37199775582934
Со сглаживанием перемешанный: -172.30526924148177


In [54]:
def perplexity_smoothed(model, tokens, alpha):
    lp = logprob_smoothed(model, tokens, alpha)
    N  = len(tokens) - 1
    return math.exp(-lp / N)

for a in (1.0, 0.1, 0.01):
    ppl = perplexity_smoothed(model, test_words, a)
    print(f"{a:<5}: {ppl:.2f}")

1.0  : 9494.26
0.1  : 4634.20
0.01 : 3277.51
